# 02 — Pembersihan Data
### Milestone M2 · Exploratory Data Analysis (KMTI21133)

| | |
|---|---|
| **Nama tim** | DANVA |
| **Anggota** | Diyanatul Fauziyah, Nasya Awaliyah, Raveska Hinayah, Asysyifa Ainaya Zahra |
| **Topik proyek** | Analisis Pola Penjualan Supermarket Berdasarkan Produk, Kota, Pelanggan, dan Metode Pembayaran |
| **Tanggal** | 15 November 2026 |

**Notebook ini mengisi:** dimensi **Justifikasi keputusan cleaning** (30), **Ketepatan implementasi** (20), dan **Dokumentasi dan keterlacakan** (15) pada Rubrik M2.

Dimensi dengan bobot terbesar di M2 bukan kode yang benar, melainkan **alasan** di balik
setiap keputusan. Untuk setiap langkah, tuliskan tiga hal: apa yang dilakukan, mengapa,
dan apa dampaknya terhadap data.

Urutan langkah di notebook ini harus sama dengan urutan di `cleaning_log.md`.

---
> **Sebelum mengumpulkan:** jalankan **Kernel → Restart & Run All**. Notebook yang tidak dapat
> berjalan dari awal sampai akhir dianggap tidak memenuhi dimensi *ketepatan implementasi*.

In [1]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 10,
    'axes.titlesize': 11, 'axes.titleweight': 'bold',
    'axes.grid': True, 'grid.alpha': 0.25,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
})

DATA  = '#0E7490'   # warna utama
SOROT = '#C2410C'   # penyorot
REDUP = '#5B6672'

# Path relatif terhadap folder notebooks/ -- JANGAN diganti menjadi path absolut.
AKAR      = Path('..')
RAW       = AKAR / 'data' / 'raw'
PROCESSED = AKAR / 'data' / 'processed'
FIGURES   = AKAR / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## 0. Muat ulang data mentah

Selalu mulai dari `data/raw`, bukan dari hasil pembersihan sebelumnya. Dengan begitu
notebook ini dapat dijalankan ulang kapan saja dan memberi hasil yang sama.

In [2]:
# Tidak perlu mount Google Drive.
# Notebook membaca dataset simulasi dari folder proyek DANVA.

Mounted at /content/drive


In [3]:
# Cari folder proyek secara fleksibel agar dapat dijalankan dari Jupyter maupun Colab.
kandidat_akar = [Path.cwd(), Path.cwd().parent]
akar_proyek = None
berkas = None

for akar in kandidat_akar:
    folder_simulasi = akar / 'data' / 'simulated'
    if folder_simulasi.exists():
        pilihan = sorted(folder_simulasi.glob('supermarket_sales_dirty*.csv'))
        if pilihan:
            akar_proyek = akar
            berkas = pilihan[0]
            break

if berkas is None:
    raise FileNotFoundError(
        "Dataset tidak ditemukan. Pastikan file supermarket_sales_dirty.csv "
        "berada di folder data/simulated/ dalam repositori DANVA."
    )

AKAR = akar_proyek
RAW = AKAR / 'data' / 'raw'
SIMULATED = AKAR / 'data' / 'simulated'
PROCESSED = AKAR / 'data' / 'processed'
FIGURES = AKAR / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

BERKAS = berkas
df_raw = pd.read_csv(BERKAS)
df = df_raw.copy()

print("Sumber data:", BERKAS)
print("Ukuran data awal:", df_raw.shape)

log = []
def catat(langkah, alasan, sebelum, sesudah):
    log.append({
        'langkah': langkah,
        'alasan': alasan,
        'baris_sebelum': sebelum,
        'baris_sesudah': sesudah,
        'selisih': sesudah - sebelum
    })
    print(f'[{len(log):02d}] {langkah}: {sebelum} -> {sesudah} baris ({sesudah-sebelum:+d})')

baris awal: 1008



## Langkah 1 — Membuat salinan dataset

**Apa:** Membuat salinan dataset simulasi menggunakan `df.copy()` ke variabel `df_clean`.

**Mengapa:** Dataset simulasi asli harus tetap dipertahankan agar dapat digunakan sebagai bukti kondisi data sebelum proses cleaning. Semua perubahan dilakukan pada salinan sehingga data awal tidak berubah.

**Alternatif yang dipertimbangkan:** Membersihkan dataset langsung pada variabel `df`. Alternatif ini tidak dipilih karena dapat mengubah dataset awal dan menyulitkan perbandingan sebelum dan sesudah cleaning.


In [4]:
n0 = len(df)

# Membuat salinan kerja; df_raw tetap dipertahankan sebagai data simulasi awal.
df_clean = df.copy()

catat(
    'Membuat salinan dataset',
    'Seluruh perubahan dilakukan pada salinan agar dataset simulasi asli tidak berubah.',
    n0,
    len(df_clean)
)

[01] ____: 1008 -> 1008 baris (+0)


In [5]:
# Bukti dampak langkah 2: distribusi Total sebelum dan sesudah penghapusan duplikat.
KOL = 'Total'
bins = np.histogram_bin_edges(df_raw[KOL].dropna(), bins=30)

fig, ax = plt.subplots(1, 2, figsize=(10, 3.4), sharex=True, sharey=True)
ax[0].hist(df_raw[KOL].dropna(), bins=bins, color=REDUP)
ax[0].set_title('Sebelum cleaning')
ax[0].set_xlabel('Total transaksi')
ax[0].set_ylabel('Frekuensi')

ax[1].hist(df_clean[KOL].dropna(), bins=bins, color=DATA)
ax[1].set_title('Sesudah menghapus duplikat')
ax[1].set_xlabel('Total transaksi')

plt.tight_layout()
plt.show()

In [6]:
print("Salinan dataset berhasil dibuat.")
print("Jumlah baris:", len(df_clean))
print("Jumlah kolom:", len(df_clean.columns))

Salinan dataset berhasil dibuat.
Jumlah baris: 1008
Jumlah kolom: 17



## Langkah 2 — Menghapus baris duplikat

**Apa:** Menghapus baris transaksi yang memiliki seluruh nilai kolom sama menggunakan `drop_duplicates()`.

**Mengapa:** Baris duplikat dapat menyebabkan transaksi dihitung lebih dari satu kali sehingga menghasilkan ringkasan statistik dan analisis penjualan yang tidak akurat.

**Alternatif yang dipertimbangkan:** Menghapus data berdasarkan `Invoice ID` saja. Alternatif ini tidak langsung dipilih karena perlu dipastikan terlebih dahulu apakah setiap Invoice ID memang harus unik dan apakah pengulangan tersebut merupakan duplikasi seluruh transaksi.


In [7]:
n0 = len(df_clean)
jumlah_duplikat = int(df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates().copy()

catat(
    'Menghapus baris duplikat',
    f'Menghapus {jumlah_duplikat} baris yang identik pada seluruh kolom agar transaksi tidak dihitung ganda.',
    n0,
    len(df_clean)
)

[02] ____: 1008 -> 1008 baris (+0)


In [8]:
print("Baris sebelum:", log[-1]['baris_sebelum'])
print("Baris sesudah:", log[-1]['baris_sesudah'])
print("Baris duplikat dihapus:", -log[-1]['selisih'])
print("Duplikasi tersisa:", df_clean.duplicated().sum())

Baris sebelum: 1008
Baris sesudah: 1000
Baris duplikat dihapus: 8



## Langkah 3 — Menyeragamkan kategori dan menghapus spasi

**Apa:** Menghapus spasi berlebih pada kolom teks dan menyeragamkan penulisan kategori pada kolom Payment.

**Mengapa:** Perbedaan huruf besar, huruf kecil, dan spasi dapat membuat kategori yang sebenarnya sama terbaca sebagai kategori berbeda. Hal ini dapat mengganggu pengelompokan dan perhitungan jumlah transaksi.

**Alternatif yang dipertimbangkan:** Menghapus baris yang memiliki kategori tidak konsisten. Alternatif ini tidak dipilih karena informasi transaksi masih dapat digunakan setelah format teks diperbaiki.


In [9]:
n0 = len(df_clean)

# Rapikan spasi pada seluruh kolom teks.
kolom_teks = df_clean.select_dtypes(include='object').columns
for kolom in kolom_teks:
    df_clean[kolom] = df_clean[kolom].str.strip()

# Seragamkan label metode pembayaran.
peta_payment = {
    'cash': 'Cash',
    'ewallet': 'Ewallet',
    'e-wallet': 'Ewallet',
    'credit card': 'Credit card'
}
df_clean['Payment'] = (
    df_clean['Payment'].str.lower().map(peta_payment)
)

catat(
    'Menyeragamkan kategori dan spasi',
    'Menghapus spasi berlebih dan menyatukan variasi label Payment agar kategori konsisten.',
    n0,
    len(df_clean)
)

[03] ____: 1008 -> 1008 baris (+0)


In [10]:
print("Kategori Payment setelah cleaning:")
display(df_clean['Payment'].value_counts(dropna=False))

print("Kategori City setelah cleaning:")
display(df_clean['City'].value_counts(dropna=False))

Kategori Payment setelah cleaning:


,count
Payment,
Ewallet,347
Cash,344
Credit card,309


Kategori City setelah cleaning:


,count
City,
Yangon,340
Mandalay,332
Naypyitaw,328



## Langkah 4 — Memperbaiki nilai Rating tidak valid

**Apa:** Mengubah kolom Rating menjadi numerik dan mengganti nilai yang berada di luar rentang 1 sampai 10 menjadi nilai kosong (NaN).

**Mengapa:** Rating merupakan penilaian pelanggan dengan rentang nilai 1 sampai 10. Nilai di luar rentang tersebut tidak valid dan dapat mengganggu analisis statistik.

**Alternatif yang dipertimbangkan:** Menghapus seluruh baris yang memiliki Rating tidak valid. Alternatif ini tidak dipilih karena informasi transaksi lainnya masih dapat digunakan meskipun nilai Rating bermasalah.


In [11]:
n0 = len(df_clean)
rating_sebelum = pd.to_numeric(df_clean['Rating'], errors='coerce')
jumlah_rating_invalid = (
    rating_sebelum.notna() & ~rating_sebelum.between(1, 10)
).sum()

df_clean['Rating'] = rating_sebelum
df_clean.loc[~df_clean['Rating'].between(1, 10), 'Rating'] = np.nan

catat(
    'Menangani Rating tidak valid',
    f'{jumlah_rating_invalid} nilai Rating di luar rentang 1–10 diubah menjadi NaN; data transaksi lainnya dipertahankan.',
    n0,
    len(df_clean)
)

print("Rating tidak valid yang diubah menjadi kosong:", int(jumlah_rating_invalid))
print("Jumlah Rating kosong setelah cleaning:", int(df_clean['Rating'].isna().sum()))

Rating tidak valid yang diubah menjadi kosong: 6
Jumlah Rating kosong setelah cleaning: 24



## Langkah 5 — Memperbaiki tipe data Quantity

**Apa:** Mengonversi kolom Quantity menjadi numerik. Nilai yang tidak dapat dikonversi, seperti teks "dua", diubah menjadi NaN.

**Mengapa:** Quantity menunjukkan jumlah barang yang dibeli dan digunakan dalam perhitungan total transaksi. Kolom ini harus bertipe numerik agar dapat dianalisis dan dihitung dengan benar.

**Alternatif yang dipertimbangkan:** Mengganti teks "dua" menjadi angka 2 secara otomatis. Alternatif ini tidak dipilih karena tidak semua teks yang ditemukan dapat dipastikan maknanya tanpa verifikasi.


In [12]:
n0 = len(df_clean)
quantity_sebelum = df_clean['Quantity'].copy()

df_clean['Quantity'] = pd.to_numeric(
    df_clean['Quantity'],
    errors='coerce'
)

jumlah_quantity_invalid = (
    quantity_sebelum.notna() & df_clean['Quantity'].isna()
).sum()

catat(
    'Mengonversi Quantity menjadi numerik',
    f'{jumlah_quantity_invalid} nilai Quantity yang tidak dapat dikonversi diubah menjadi NaN karena maknanya tidak dapat dipastikan.',
    n0,
    len(df_clean)
)

print("Quantity tidak dapat dikonversi:", int(jumlah_quantity_invalid))
print("Tipe data Quantity:", df_clean['Quantity'].dtype)
print("Jumlah Quantity kosong:", int(df_clean['Quantity'].isna().sum()))

Quantity tidak dapat dikonversi: 4
Tipe data Quantity: float64
Jumlah Quantity kosong: 4



## Langkah 6 — Mengubah tipe data Date dan Time

**Apa:** Mengonversi kolom Date dan Time dari teks menjadi tipe data tanggal dan waktu menggunakan `pd.to_datetime()`.

**Mengapa:** Format tanggal dan waktu yang masih berupa teks menyulitkan pengolahan berdasarkan periode, tanggal, dan jam transaksi. Konversi membuat data lebih mudah dianalisis.

**Alternatif yang dipertimbangkan:** Membiarkan Date dan Time dalam bentuk teks. Alternatif ini tidak dipilih karena membatasi analisis berdasarkan urutan waktu dan periode transaksi.


In [13]:
n0 = len(df_clean)

date_sebelum = df_clean['Date'].copy()
time_sebelum = df_clean['Time'].copy()

df_clean['Date'] = pd.to_datetime(
    df_clean['Date'], format='%m/%d/%Y', errors='coerce'
)
df_clean['Time'] = pd.to_datetime(
    df_clean['Time'], format='%H:%M', errors='coerce'
)

jumlah_date_gagal = int(df_clean['Date'].isna().sum() - date_sebelum.isna().sum())
jumlah_time_gagal = int(df_clean['Time'].isna().sum() - time_sebelum.isna().sum())

catat(
    'Mengonversi Date dan Time',
    f'Mengubah Date dan Time menjadi tipe tanggal/waktu; gagal konversi baru: Date {jumlah_date_gagal}, Time {jumlah_time_gagal}.',
    n0,
    len(df_clean)
)

print("Jumlah Date kosong/gagal:", int(df_clean['Date'].isna().sum()))
print("Jumlah Time kosong/gagal:", int(df_clean['Time'].isna().sum()))
display(df_clean[['Date', 'Time']].head())

Jumlah Date gagal dikonversi: 0
Jumlah Time gagal dikonversi: 0


,Date,Time
0,2019-01-05,1900-01-01 13:08:00
1,2019-03-08,1900-01-01 10:29:00
2,2019-03-03,1900-01-01 13:23:00
3,2019-01-27,1900-01-01 20:33:00
4,2019-02-08,1900-01-01 10:37:00



## Langkah 7 — Validasi dan menyimpan dataset bersih

**Apa:** Memeriksa kembali hasil cleaning, termasuk jumlah baris, duplikasi, nilai kosong, dan tipe data. Setelah itu, menyimpan dataset bersih ke `data/processed/data_bersih.csv`.

**Mengapa:** Validasi diperlukan untuk memastikan proses cleaning berjalan sesuai rencana dan hasilnya dapat digunakan untuk analisis berikutnya.

**Alternatif yang dipertimbangkan:** Menggunakan dataset langsung tanpa menyimpan hasil cleaning. Alternatif ini tidak dipilih karena proses analisis berikutnya membutuhkan dataset bersih yang konsisten dan dapat digunakan kembali.


In [15]:
print("Ukuran dataset setelah cleaning:", df_clean.shape)
print("\nDuplikasi tersisa:", int(df_clean.duplicated().sum()))

print("\nNilai kosong setiap kolom:")
display(df_clean.isna().sum().to_frame('jumlah_kosong'))

print("\nTipe data:")
display(df_clean.dtypes.to_frame('tipe_data'))

Ukuran dataset setelah cleaning: (1000, 17)

Duplikasi tersisa: 0

Nilai kosong setiap kolom:


,0
Invoice ID,0
Branch,0
City,0
Customer type,0
Gender,0
Product line,0
Unit price,0
Quantity,4
Tax 5%,0
Total,0



Tipe data:


,0
Invoice ID,object
Branch,object
City,object
Customer type,object
Gender,object
Product line,object
Unit price,float64
Quantity,float64
Tax 5%,float64
Total,float64



Dataset bersih berhasil disimpan: ../data/processed/data_bersih.csv


## Penanganan nilai yang hilang

Keputusan ini dinilai paling ketat. Dasarkan pada **mekanisme** kekosongan yang Anda
temukan di `01_quality.ipynb`, bukan pada kemudahan.

| Mekanisme | Artinya | Penanganan yang biasanya sesuai |
|---|---|---|
| MCAR | kekosongan acak sepenuhnya | boleh dibuang; bias kecil |
| MAR | kekosongan dijelaskan kolom lain yang teramati | imputasi dengan prediktor, atau analisis terpisah |
| MNAR | kekosongan bergantung pada nilai yang tidak terlihat | **jangan** diisi rata-rata; laporkan dan batasi klaim |

Perlu diingat: MAR dan MNAR **tidak dapat dibedakan** hanya dari data. Nyatakan asumsi
yang Anda pakai secara terbuka.

In [ ]:
# Keputusan penanganan nilai hilang:
# Rating yang sejak awal kosong tetap dipertahankan sebagai NaN.
# Rating di luar rentang 1–10 dan Quantity yang tidak dapat dikonversi
# juga diubah menjadi NaN, bukan diisi dengan mean/median atau ditebak.
#
# Alasan:
# Dataset tidak menyediakan informasi yang cukup untuk mengetahui nilai
# Rating atau Quantity yang sebenarnya. Mengisi nilai secara sembarang dapat
# mengubah distribusi dan menghasilkan informasi transaksi yang tidak benar.
#
# Asumsi mekanisme:
# Pola missingness telah diperiksa pada 01_quality.ipynb. Untuk nilai yang
# tidak dapat dipastikan mekanismenya, kami tidak mengklaim MCAR/MAR/MNAR.
# Nilai kosong dilaporkan dan dipertahankan agar keterbatasan data transparan.

print("Nilai kosong yang dipertahankan:")
display(df_clean.isna().sum().loc[lambda s: s > 0].to_frame('jumlah_kosong'))

## Penanganan data pribadi

Wajib dilakukan **sebelum** data disimpan ke `data/processed`.

In [ ]:
import hashlib

def samarkan(s):
    """Hash satu arah; nilai asli tidak dapat dikembalikan."""
    return hashlib.sha256(str(s).encode()).hexdigest()[:12]

# Pemeriksaan kolom: dataset supermarket ini tidak memuat nama, nomor telepon,
# alamat, atau identitas pribadi langsung lainnya.
# Invoice ID merupakan pengenal transaksi, bukan nama atau kontak pelanggan.
kolom_pribadi = [
    kolom for kolom in df_clean.columns
    if kolom.lower() in {'nama', 'name', 'nomor_hp', 'no_hp', 'phone',
                         'alamat', 'address', 'email', 'email_address'}
]

if kolom_pribadi:
    df_clean = df_clean.drop(columns=kolom_pribadi)

print("Kolom yang dihapus karena berisi identitas pribadi:", kolom_pribadi)
print("Kolom dataset setelah pemeriksaan:")
print(df_clean.columns.tolist())

## Validasi akhir

Tuliskan sebagai `assert`. Bila suatu saat data atau kode berubah dan hasilnya rusak,
notebook akan berhenti di sini — bukan diam-diam menghasilkan angka yang salah.

In [ ]:
# Validasi kualitas hasil cleaning.
assert df_clean.duplicated().sum() == 0, 'Masih ada baris duplikat.'
assert df_clean['Payment'].dropna().isin(
    ['Cash', 'Ewallet', 'Credit card']
).all(), 'Masih ada label Payment yang tidak seragam.'
assert df_clean['Rating'].dropna().between(1, 10).all(), 'Rating di luar rentang 1–10.'
assert pd.api.types.is_numeric_dtype(df_clean['Quantity']), 'Quantity belum numerik.'
assert pd.api.types.is_datetime64_any_dtype(df_clean['Date']), 'Date belum bertipe tanggal.'
assert pd.api.types.is_datetime64_any_dtype(df_clean['Time']), 'Time belum bertipe waktu.'
assert len(df_clean) <= len(df_raw), 'Jumlah baris hasil cleaning melebihi data awal.'

print(f'LOLOS. {len(df_clean)} baris x {df_clean.shape[1]} kolom siap dianalisis.')

## Simpan hasil

In [ ]:
PROCESSED.mkdir(parents=True, exist_ok=True)
KELUARAN = PROCESSED / 'data_bersih.csv'
df_clean.to_csv(KELUARAN, index=False)

print('Dataset bersih tersimpan:', KELUARAN)

ringkas = pd.DataFrame(log)
display(ringkas)

print(
    f"Total baris: {len(df_raw)} -> {len(df_clean)} "
    f"({(len(df_clean)-len(df_raw))/len(df_raw)*100:+.1f}%)"
)

In [ ]:
# Salin keluaran sel ini ke cleaning_log.md
baris = [
    '# Cleaning Log', '',
    f'Sumber   : `{BERKAS.name}` ({len(df_raw)} baris)',
    f'Keluaran : `{KELUARAN.name}` ({len(df_clean)} baris)', '',
    '| # | Langkah | Alasan | Baris sebelum | Baris sesudah |',
    '|---|---------|--------|---------------|---------------|'
]
for i, r in enumerate(log, 1):
    alasan = str(r['alasan']).replace('|', '\\|')
    baris.append(
        f"| {i} | {r['langkah']} | {alasan} | "
        f"{r['baris_sebelum']} | {r['baris_sesudah']} |"
    )

teks_log = '\n'.join(baris)
print(teks_log)

# Simpan juga otomatis ke folder proyek agar tidak perlu menyalin manual.
(AKAR / 'cleaning_log.md').write_text(teks_log + '\n', encoding='utf-8')
print("\nCleaning log berhasil diperbarui:", AKAR / 'cleaning_log.md')

## Checklist sebelum mengumpulkan

- [ ] Setiap langkah memiliki **apa**, **mengapa**, dan **alternatif yang dipertimbangkan**.
- [ ] Minimal satu langkah disertai bukti dampak berupa perbandingan sebelum–sesudah.
- [ ] Panel sebelum–sesudah memakai sumbu dan lebar bin yang identik.
- [ ] Penanganan nilai hilang didasarkan pada mekanismenya, bukan kemudahan.
- [ ] Data pribadi sudah ditangani **dan** diverifikasi.
- [ ] Ada blok `assert` validasi akhir.
- [ ] `cleaning_log.md` sudah diperbarui dari keluaran sel terakhir.
- [ ] `data/raw` tidak berubah.